 # How Regex pattern catches the PII and the credentials before it reaches to the LLM

In [3]:
import re
import os
from typing import Optional
from datetime import datetime
from dataclasses import dataclass, field

In [39]:
@dataclass
class GaurdrailResult:
    passed :bool
    layer :str
    reason :str
    score: Optional[float] = None
    redacted : Optional[str] = None
    timestamp :str = field(default_factory = lambda: datetime.utcnow().isoformat())

    def __str__(self):
        status = "PASS" if self.passed else "BLOCK"
        score = f"score = {self.score:.3f}"  if self.score is not None else "" 
        return f"{self.layer} -  {status} {score} : {self.reason}"
    





In [40]:
r = GaurdrailResult(passed=True, layer = "L1-Regex", reason = "No PII information detected")
print(r)

L1-Regex -  PASS  : No PII information detected


/var/folders/2h/sjyl9z_s50v8pl2446pvqjrw0000gn/T/ipykernel_9871/1689136956.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp :str = field(default_factory = lambda: datetime.utcnow().isoformat())


In [41]:
_PATTERNS = {
    # ── Identifiers ──────────────────────────────────────────────
    "email":        re.compile(r"[\w.+-]+@[\w-]+\.\w{2,}", re.I),
    "phone_us":     re.compile(r"\b(\+?1[-.\s]?)?\(?\d{3}\)?[-.\s]\d{3}[-.\s]\d{4}\b"),
    "ssn":          re.compile(r"\b\d{3}-\d{2}-\d{4}\b"),
    "credit_card":  re.compile(
        r"\b(?:4\d{12}(?:\d{3})?|5[1-5]\d{14}|3[47]\d{13}|"
        r"3(?:0[0-5]|[68]\d)\d{11}|6(?:011|5\d{2})\d{12})\b"
    ),
    "ip_address":   re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b"),

    # ── Credentials ──────────────────────────────────────────────
    "password_kw":  re.compile(r"\b(password|passwd|secret|api[_-]?key|token|bearer)\s*[=:\"\']\s*\S+", re.I),
    "aws_key":      re.compile(r"\b(AKIA|ASIA|AROA)[A-Z0-9]{16}\b"),
    "jwt":          re.compile(r"\beyJ[A-Za-z0-9_-]+\.[A-Za-z0-9_-]+\.[A-Za-z0-9_-]+\b"),

    # ── Harmful keywords ─────────────────────────────────────────
    "harm_keywords": re.compile(
        r"\b(make\s+a\s+bomb|synthesize\s+drugs|child\s+exploitation)\b", re.I
    ),
}

_REDACT_MAP = {
    "email": "[EMAIL]", "phone_us": "[PHONE]", "ssn": "[SSN]",
    "credit_card": "[CREDIT_CARD]", "ip_address": "[IP_ADDRESS]",
    "password_kw": "[CREDENTIAL]", "aws_key": "[AWS_KEY]", "jwt": "[JWT]",
}

print(f"Loaded {len(_PATTERNS)} patterns")
for name, pat in _PATTERNS.items():
    print(f"  {name:15s} → {pat.pattern[:55]}...")

Loaded 9 patterns
  email           → [\w.+-]+@[\w-]+\.\w{2,}...
  phone_us        → \b(\+?1[-.\s]?)?\(?\d{3}\)?[-.\s]\d{3}[-.\s]\d{4}\b...
  ssn             → \b\d{3}-\d{2}-\d{4}\b...
  credit_card     → \b(?:4\d{12}(?:\d{3})?|5[1-5]\d{14}|3[47]\d{13}|3(?:0[0...
  ip_address      → \b(?:\d{1,3}\.){3}\d{1,3}\b...
  password_kw     → \b(password|passwd|secret|api[_-]?key|token|bearer)\s*[...
  aws_key         → \b(AKIA|ASIA|AROA)[A-Z0-9]{16}\b...
  jwt             → \beyJ[A-Za-z0-9_-]+\.[A-Za-z0-9_-]+\.[A-Za-z0-9_-]+\b...
  harm_keywords   → \b(make\s+a\s+bomb|synthesize\s+drugs|child\s+exploitat...


In [42]:
for name, pattern in _PATTERNS.items():
        print(name,pattern)

email re.compile('[\\w.+-]+@[\\w-]+\\.\\w{2,}', re.IGNORECASE)
phone_us re.compile('\\b(\\+?1[-.\\s]?)?\\(?\\d{3}\\)?[-.\\s]\\d{3}[-.\\s]\\d{4}\\b')
ssn re.compile('\\b\\d{3}-\\d{2}-\\d{4}\\b')
credit_card re.compile('\\b(?:4\\d{12}(?:\\d{3})?|5[1-5]\\d{14}|3[47]\\d{13}|3(?:0[0-5]|[68]\\d)\\d{11}|6(?:011|5\\d{2})\\d{12})\\b')
ip_address re.compile('\\b(?:\\d{1,3}\\.){3}\\d{1,3}\\b')
password_kw re.compile('\\b(password|passwd|secret|api[_-]?key|token|bearer)\\s*[=:\\"\\\']\\s*\\S+', re.IGNORECASE)
aws_key re.compile('\\b(AKIA|ASIA|AROA)[A-Z0-9]{16}\\b')
jwt re.compile('\\beyJ[A-Za-z0-9_-]+\\.[A-Za-z0-9_-]+\\.[A-Za-z0-9_-]+\\b')
harm_keywords re.compile('\\b(make\\s+a\\s+bomb|synthesize\\s+drugs|child\\s+exploitation)\\b', re.IGNORECASE)


In [43]:
def regex_filter(text:str, redact:bool = True) -> GaurdrailResult:
    
    redacted_text = text
    triggered = []

    for name, pattern in _PATTERNS.items():
        if pattern.search(text):
            triggered.append(name)
            if redact and name in _REDACT_MAP:
                redacted_text = pattern.sub(_REDACT_MAP[name],redacted_text)

    if triggered:
        return GaurdrailResult(passed=False,layer="L1-Regex",reason = f"Triggered Reason: {' , '.join(triggered)}",
                               redacted = redacted_text if redact else None )
    

    return GaurdrailResult(passed=True,layer="L1-Regex",reason = "No Pattern Matched" )


In [44]:
result = regex_filter("What is the best way to learn AI?")
print(result)

L1-Regex -  PASS  : No Pattern Matched


/var/folders/2h/sjyl9z_s50v8pl2446pvqjrw0000gn/T/ipykernel_9871/1689136956.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp :str = field(default_factory = lambda: datetime.utcnow().isoformat())


In [45]:
result = regex_filter("Please send report to abc@gamil.com")
print(result)

L1-Regex -  BLOCK  : Triggered Reason: email


/var/folders/2h/sjyl9z_s50v8pl2446pvqjrw0000gn/T/ipykernel_9871/1689136956.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp :str = field(default_factory = lambda: datetime.utcnow().isoformat())


In [46]:
result = regex_filter("My SSN number is 123-45-7689")
print(result)

L1-Regex -  BLOCK  : Triggered Reason: ssn


/var/folders/2h/sjyl9z_s50v8pl2446pvqjrw0000gn/T/ipykernel_9871/1689136956.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp :str = field(default_factory = lambda: datetime.utcnow().isoformat())
